In [ ]:
# SummarizationMiddleware

In [7]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv(override=True)

chat_model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", )
summary_model = init_chat_model(model="deepseek:deepseek-v4-flash",
                                profile={"max_input_tokens": 1_000_000})

agent = create_agent(
    model=chat_model,
    middleware=[SummarizationMiddleware(
        model=summary_model,
        trigger=[
            ("tokens", 100),
            ("messages", 6),
            ("fraction", 0.002)
        ],
        keep=("messages", 2),
    )]
)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

response = agent.invoke({"messages": messages})

for msg in response.get("messages"):
    msg.pretty_print()

================================ System Message ================================

你是个非常友好的AI助手
================================ Human Message =================================

你好啊，我是老王，你是谁？
================================== Ai Message ==================================

你好老王，我是小王
================================ Human Message =================================

好的小王，很高兴认识你
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，我是在开玩笑，没别的意思 😄 很高兴认识你，老王！有什么想聊的吗？


In [8]:
agent = create_agent(
    model=chat_model,
    middleware=[SummarizationMiddleware(
        model=summary_model,
        trigger=[
            ("tokens", 100),
            ("messages", 6),
            ("fraction", 0.002)
        ],
        keep=("messages", 2),
        summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
    )]
)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

response = agent.invoke({"messages": messages})

for msg in response.get("messages"):
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

根据对话记录，用户“老王”与AI助手“小王”相互问候并表达认识意愿。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

没什么意思，逗你玩的——认识你的机会当然还在，只是想故意卖个关子。你好，我是小王，很高兴认识你。
